# NumPy and PyTorch Foundations

Debug array shapes and broadcasting, assemble image tensors, and train a linear regressor.

Complete the implementation tasks before consulting the reference notebook.


### Optional: Google Colab setup

Open this notebook in [Google Colab](https://colab.research.google.com/), then run the next cell first. If prompted, upload the complete `basicrl-study.zip` material bundle. It prepares the matching topic directory and, for robotics, the headless rendering libraries. Continue with the dependency-install cell below.

Local Jupyter skips the Colab-only operations. The step-by-step guide, runtime choices, and result-saving instructions are in `docs/COLAB.md` in the material bundle.


In [ ]:
# Optional Colab bootstrap. Local Jupyter leaves the working directory unchanged.
from pathlib import Path, PurePosixPath
import io
import runpy
import zipfile

try:
    import google.colab
except ImportError:
    print("Local Jupyter: continue from this notebook's own directory.")
else:
    from google.colab import files
    material_root = Path("/content/basicrl-study")
    if not (material_root / "scripts/colab_setup.py").is_file():
        print("Upload the complete basicrl-study.zip material bundle.")
        uploaded = files.upload()
        archives = [name for name in uploaded if name.lower().endswith(".zip")]
        if len(archives) != 1:
            raise ValueError("Choose exactly one basicrl-study.zip material bundle.")
        with zipfile.ZipFile(io.BytesIO(uploaded[archives[0]])) as archive:
            names = archive.namelist()
            for name in names:
                parts = PurePosixPath(name).parts
                if (not parts or parts[0] != "basicrl-study" or ".." in parts
                        or PurePosixPath(name).is_absolute() or "\\" in name):
                    raise ValueError("Use the material bundle produced by export_release.py.")
            required = {"basicrl-study/scripts/colab_setup.py",
                        "basicrl-study/requirements/basics.txt",
                        "basicrl-study/assignments/00-numpy-pytorch/exercises/notebook.ipynb"}
            if not required.issubset(names):
                raise ValueError("The material bundle is incomplete or outdated.")
            archive.extractall(material_root.parent)
    setup = runpy.run_path(str(material_root / "scripts/colab_setup.py"))
    setup["prepare_notebook"](material_root, "00-numpy-pytorch", "exercises")


## Dependencies

Run the following install cell in Colab, or when setting up a local environment. In local Jupyter, launch the kernel from this notebook's directory so `q1.py` and `q2.py` resolve to the exercise or solution copy you selected. If installation asks for a session restart, restart and begin again from the Colab setup cell.


In [ ]:
%pip install -r ../../../requirements/basics.txt


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

# Reload code in q1 and q2 modules without having to restart the kernel
%load_ext autoreload
%autoreload 2

# Set seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

### Q1: Shapes and Sizes

In this first question, we hope to get you familiar thinking about and working with shapes in both Numpy and PyTorch. The most common pain points we see in these exercises are related to debugging shape and sizing issues. To help you get familiar with this, we've provided some buggy code that needs to be debugged.

There are 3 functions to modify in this section, located in `q1.py`. Below, `calculate_distances` is called with some inputs and there are two assert statements that check for the correct shape and correct numbers. Please modify `calculate_distances` in `q1.py` to pass the asserts. You should not need to modify the inputs. The bugs are also simple enough to be fixed in-line - you should not need to add many lines of code.

To debug, we suggest printing out each intermediate step of your code to check that the shapes are what you expect them to be. If you are confused about the output of a print statement, check out the documentation or ask for help.

In [ ]:
import q1

x = np.array([[0, 0], [1, 0], [2, 0]])
y = np.array([[3, 4], [3, 0], [-2, 0]])
n, d = x.shape

distance = q1.calculate_distances(x, y)
assert distance.shape == (n, 1), f"Wrong shape: {distance.shape} != {(n, 1)}"
assert np.array_equal(distance, np.array([[5], [2], [4]])), f"Wrong values: {distance}"

Once you have fixed `calculate_distances` and the assert statements are passing, try fixing function `combine_squares` below. Now you are working with PyTorch tensors but they are very similar to Numpy arrays. Use the same debugging principles as before.

In [ ]:
square_1 = torch.tensor([[0 , 1 ], [4 , 5 ]])
square_2 = torch.tensor([[2 , 3 ], [6 , 7 ]])
square_3 = torch.tensor([[8 , 9 ], [12, 13]])
square_4 = torch.tensor([[10, 11], [14, 15]])
n, _ = square_1.shape

combined_square = q1.combine_squares(square_1, square_2, square_3, square_4)
assert combined_square.shape == (2 * n, 2 * n), f"Wrong shape: {combined_square.shape} != {(2 * n, 2 * n)}"
assert torch.equal(combined_square, torch.tensor([[0, 1, 2, 3], [4, 5, 6, 7], [8, 9, 10 ,11], [12, 13, 14, 15]])), f"Wrong values: {combined_square}"

Now you will combine your knowledge from the previous two functions and implement `video_to_filmstrip`. We omit the assert statements here so you should write your own to verify the correctness of your code. For reference, the solution is 3 lines of code. Your code should be vectorized meaning do not use any for loops and only use Numpy/PyTorch functions.

In [ ]:
video_frames = []
for i in range(10):
    video_frames.append(np.ones((1080, 1920, 3)) * i)

filmstrip = q1.video_to_filmstrip(video_frames)

### Q2: A Simple Regressor

This second question will hopefully refresh your memory of how to train a model in PyTorch and give you a general idea of what coding exercises will look like in these exercises. 

In this question, we are testing your ability to solve a simple regression problem. First we generate some train and test data. This should not be modified.

In [ ]:
# Generate train/test data
data_size = 500
X = np.random.rand(data_size, 1)
Y = -4 * X + 2 + np.random.randn(data_size, 1)

test_split = int(data_size * 0.8)
x_train, x_test = X[:test_split], X[test_split:]
y_train, y_test = Y[:test_split], Y[test_split:]

x_train, y_train = torch.tensor(x_train).float(), torch.tensor(y_train).float()
x_test, y_test = torch.tensor(x_test).float(), torch.tensor(y_test).float()

Now your task is to implement a simple linear regressor. Refer to script `q2.py` and implement the TODOs. You will need to implement the `forward()` pass, specify a loss function and optimizer, and write a training loop that includes checkpointing logic to save the best model.

In [ ]:
import q2
checkpoint_path = q2.get_checkpoint_path()
model = q2.LinearRegression()
loss_fn, optimizer = q2.create_loss_and_optimizer(model)
q2.train(x_train, y_train, model, loss_fn, optimizer, checkpoint_path, num_epochs=1000)

You should also implement `load_model_checkpoint` so you can load your best model to evaluate. This makes checkpoint evaluation reproducible. You can also save information such as the optimizer state and epochs to resume training if your training is interrupted.

In [ ]:
model = q2.load_model_checkpoint(checkpoint_path)

Below we provide code snippets to visualize the performance of your model on the training and test data. You do not need to modify this code - use this as a sanity check to make sure your training code is working. The test plot will be saved to `q2_test.png` for later comparison.

In [ ]:
# Plot training performance 
plt.scatter(x_train, y_train, s=10)
with torch.no_grad():
  y_pred = model(x_train)
y_pred = y_pred.numpy()
plt.plot(x_train, y_pred, color='magenta')
plt.show()

In [ ]:
# Test the model
with torch.no_grad():
  y_pred = model(x_test)
  test_loss = loss_fn(y_pred, y_test)
  print(f"Test loss: {test_loss.item()}")

In [ ]:
# Plot test performance 
plt.scatter(x_test, y_test, s=10)
with torch.no_grad():
  y_pred = model(x_test)
y_pred = y_pred.numpy()
plt.plot(x_test, y_pred, color='red')
plt.savefig('q2_test.png')
plt.show()